In [2]:
from AgentExc import AgentManager

import operator
from typing import Annotated, Any, Dict, List, Optional, Sequence, Tuple, TypedDict, Union
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
import functools
from langgraph.graph import StateGraph, END

from langchain_core.tools import tool, create_schema_from_function,convert_runnable_to_tool 
# from langchain_core.tools import Tool,create_retriever_tool



from langchain.agents import (create_openai_functions_agent, AgentExecutor, create_sql_agent,
                              create_vectorstore_agent,
                              AgentType,
                              create_openai_tools_agent,
                              create_json_chat_agent,
                              create_react_agent)
from langchain_core.messages import BaseMessage, HumanMessage,AIMessage
from langchain_openai import ChatOpenAI

from langchain_core.runnables import RunnableConfig,Runnable
import datetime


In [2]:


@tool
def check_data(data:Annotated[str, "data need to check"]):
    """Needs to check the data"""
    return data




def create_agent(llm, tools, system_message, date_time):
    prompt = ChatPromptTemplate.from_messages([
    ("system", system_message),
    MessagesPlaceholder(variable_name="messages"),MessagesPlaceholder(variable_name="agent_scratchpad")
    ])#.partial( Current_TIME = date_time)


    # agent = OpenAIMultiFunctionsAgent(
    #     llm=llm,
    #     tools=tools,
    #     prompt = prompt)

    
    agent = create_openai_tools_agent(
        llm=llm,
        tools=tools,
        prompt = prompt)


    # callback_handlers = [MyCustomCallbackHandler()]
    # callback_handlers = BaseCallbackManager(handlers=[MyCustomCallbackHandler()])


    executer = AgentManager(agent=agent, tools=tools, verbose=True,max_iterations=None,return_intermediate_steps=True,
                            #  callbacks=[StdOutCallbackHandler()],
                            #  callbacks=callback_handlers,
                             early_stopping_method = "generate",
                             handle_parsing_errors = True,
                             trim_intermediate_steps = 3,
                             include_run_info = True
                             )

    return executer


In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

In [4]:
grqo_api_key = "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"

In [5]:
import os
os.environ["GROQ_API_KEY"] = grqo_api_key

In [32]:
##openai agent
# system_prompt = ("You are an powerful assitant","Your task is to make a plan which its given by the manager.")


# name = "o3-mini-2025-01-31"
# llm = ChatOpenAI(model = name,verbose=True,cache=False)

# today_time_date = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
# memory_agent = create_agent(llm, [check_data], system_prompt, today_time_date)


# plan_agent = memory_agent._action_agent


### groq agent 
# system_prompt = "You are an powerful assitant. Your task is to make a plan which its given by the manager."
# llm = ChatGroq(temperature=0, model_name="llama-3.3-70b-versatile")
# today_time_date = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
# memory_agent = create_agent(llm, [check_data], system_prompt, today_time_date)
# plan_agent = memory_agent._action_agent


###
#groq strutured output
from langchain_groq import ChatGroq

from pydantic import BaseModel, Field
from typing import List


class Plans(BaseModel):
    '''This is used when you have to make a plan'''
    plan: List[str] = Field(..., description="List of plans what to do and how to execute the task.")


class CasualAnswer(BaseModel):
    '''This is used to make a normal back answer to the manager.'''
    msg: str = Field(..., description="Casual answer to the manager. Plans not included")


model_name="llama-3.3-70b-versatile"
# model_name = "mixtral-8x7b-32768"
llm = ChatGroq(
    model=model_name,
    temperature=0.0,
    max_retries=2,
)



from langchain_core.messages import BaseMessage, HumanMessage,AIMessage, SystemMessage


# messages = [
#     ("system", """You are an powerfull brain module of a robot. The robot is working in a store.For Casual Answer use  `CasualAnswer`.For making plan use `Plans`
#      Your task is you will recevice a command from the Manager and you make a plan how to execute that.
#      """),
# ]


messages = [
SystemMessage(content="""You are an powerfull brain module of a robot. The robot is working in a store.For Casual Answer use  `CasualAnswer`.For making plan use `Plans`
#      Your task is you will recevice a command from the Manager and you make a plan how to execute that.
#      """)
]


model_with_tools = llm.bind_tools([Plans,CasualAnswer])


In [36]:
messages.append(HumanMessage(content="restock aisle 5"))
ai_msg = model_with_tools.invoke(messages)
ai_msg

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_dg2m', 'function': {'arguments': '{"plan": ["Check the inventory levels of aisle 5", "Gather the necessary items to restock", "Move to aisle 5 and begin restocking shelves", "Ensure all items are faced and organized"]}', 'name': 'Plans'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 388, 'total_tokens': 440, 'completion_time': 0.189090909, 'prompt_time': 0.018421772, 'queue_time': 0.23352472800000001, 'total_time': 0.207512681}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_5f849c5a0b', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-26129cc1-86af-4152-9b4f-477e818f0dfd-0', tool_calls=[{'name': 'Plans', 'args': {'plan': ['Check the inventory levels of aisle 5', 'Gather the necessary items to restock', 'Move to aisle 5 and begin restocking shelves', 'Ensure all items are faced and organized']}, 'id': 'call_dg2m', 'type': 'tool_call'}]

In [41]:
output = ai_msg.tool_calls

In [46]:
output

[{'name': 'Plans',
  'args': {'plan': ['Check the inventory levels of aisle 5',
    'Gather the necessary items to restock',
    'Move to aisle 5 and begin restocking shelves',
    'Ensure all items are faced and organized']},
  'id': 'call_dg2m',
  'type': 'tool_call'}]

In [48]:
output[0]["args"]["plan"]

['Check the inventory levels of aisle 5',
 'Gather the necessary items to restock',
 'Move to aisle 5 and begin restocking shelves',
 'Ensure all items are faced and organized']

In [36]:
memory_agent

AgentManager(verbose=True, agent=RunnableMultiActionAgent(runnable=RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_to_openai_tool_messages(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'messages'], input_types={'messages': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Anno

In [37]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, FunctionMessage,AIMessageChunk
from langchain_core.agents import AgentAction

inputs = {"messages": [    ]}

In [ ]:
git checkout -b development


In [ ]:
git add .
git commit -m "Initial commit on developemt branch"


In [ ]:
q = "Hi"

# c = {"messages":[HumanMessage(content =  f"Question: {q}")]}

inputs["messages"].append(HumanMessage(content = q) )
inputs["Current_TIME"] = f"{today_time_date}"
print(inputs)
result = memory_agent.invoke(inputs)
# print(plan_agent.plan(inputs))
result

inputs["messages"].append(AIMessage(content= result["output"]))

In [20]:

inputs["messages"]

[HumanMessage(content='Hi', additional_kwargs={}, response_metadata={}),
 AIMessage(content="Hello, I'm ready to help you with your task. However, I don't see a specific plan or details from your manager. Could you please provide more information about the task you need to complete?", additional_kwargs={}, response_metadata={})]

In [ ]:
### Visual model agent to plan

In [9]:
### check on image


###
#groq strutured output
from langchain_groq import ChatGroq

from pydantic import BaseModel, Field
from typing import List


class Plans(BaseModel):
    '''This is used when you have to make a plan'''
    plan: List[str] = Field(..., description="List of plans what to do and how to execute the task.")



# model_name="llama-3.3-70b-versatile"
# model_name = "mixtral-8x7b-32768"

model_name = "llama-3.2-11b-vision-preview"
llm = ChatGroq(
    model=model_name,
    temperature=0.0,
    max_retries=2,
)



from langchain_core.messages import BaseMessage, HumanMessage,AIMessage, SystemMessage


# messages = [
#     ("system", """You are an powerfull brain module of a robot. The robot is working in a store.For Casual Answer use  `CasualAnswer`.For making plan use `Plans`
#      Your task is you will recevice a command from the Manager and you make a plan how to execute that.
#      """),
# ]


messages = [
SystemMessage(content="""You are an powerfull brain module of a robot. The robot is working in a store.
#      Your task is you will recevice a command from the Manager and you make a plan how to execute that.
#      """)
]


model_with_tools = llm.bind_tools([Plans])

In [11]:
messages = []



human = HumanMessage(content=[
                                {
                                    "type": "text",
                                    "text": """You are an powerfull brain module of a robot. The robot is working in a store.For Casual Answer use  `CasualAnswer`.For making plan use `Plans`
      Your task is you will recevice a command from the Manager and you make a plan how to execute that.restock asile 5"""
                                },
                                {
                                    "type": "image_url",
                                    "image_url": {
                                        "url": "https://c8.alamy.com/comp/2BW0RXF/orlandoflusa-5420-aisles-at-a-publix-grocery-store-with-signs-above-each-aisle-designating-what-is-contained-in-the-aisle-2BW0RXF.jpg"
                                    }
                                }
                            ])




# import base64
# def get_image(image_path):
#         with open(image_path, "rb") as image_file:
#             return base64.b64encode(image_file.read()).decode('utf-8')


# def encode_image(image_path):
#     base64_image = get_image(image_path)
#     return base64_image 

# base64_image = encode_image("sample_2.jpg")
# human = HumanMessage(content=[
#                                 {
#                                     "type": "text",
#                                     "text": """You are an powerfull brain module of a robot. The robot is working in a store.For Casual Answer use  `CasualAnswer`.For making plan use `Plans`
#       Your task is you will recevice a command from the Manager and you make a plan how to execute that.restock asile 5"""
#                                 },
#                                 {
#                                     "type": "image_url",
#                                     "image_url": {
#                                         "url": f"data:image/jpeg;base64,{base64_image}"
#                                     }
#                                 }
#                             ])





messages.append(human)
ai_msg = model_with_tools.invoke(messages)
ai_msg

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_xak6', 'function': {'arguments': '{"name":"Restock Asile 5","description":"Restock Asile 5 with necessary items","plan":["Check inventory levels","Order restock from warehouse","Restock shelves","Verify restock is complete"]}', 'name': 'Plans'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 279, 'total_tokens': 331, 'completion_time': 0.072909185, 'prompt_time': 0.018726031, 'queue_time': 1.56778563, 'total_time': 0.091635216}, 'model_name': 'llama-3.2-11b-vision-preview', 'system_fingerprint': 'fp_fa3d3d25b0', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-4a0989fd-3b3b-4e87-a305-f2700b90b666-0', tool_calls=[{'name': 'Plans', 'args': {'name': 'Restock Asile 5', 'description': 'Restock Asile 5 with necessary items', 'plan': ['Check inventory levels', 'Order restock from warehouse', 'Restock shelves', 'Verify restock is complete']}, 'id': 'call_xak6', 'ty

In [8]:
ai_msg.tool_calls[0]["args"]["plan"]

['Check inventory levels',
 'Order restock from warehouse',
 'Restock shelves',
 'Verify restock is complete']

## brain create react agent

In [ ]:
#groq strutured output
from langchain_groq import ChatGroq

from pydantic import BaseModel, Field
from typing import List


class Plans(BaseModel):
    '''This is used when you have to make a plan'''
    plan: List[str] = Field(..., description="List of plans what to do and how to execute the task.")



model_name="llama-3.3-70b-versatile"
# model_name = "mixtral-8x7b-32768"
llm = ChatGroq(
    model=model_name,
    temperature=0.0,
    max_retries=2,
)



from langchain_core.messages import BaseMessage, HumanMessage,AIMessage, SystemMessage




messages = [
SystemMessage(content="""You are an powerfull brain module of a robot. The robot is working in a store.For Casual Answer use  `CasualAnswer`.For making plan use `Plans`
#      Your task is you will recevice a command from the Manager and you make a plan how to execute that.
#      """)
]







In [15]:


import json
from typing import Annotated, List, Literal

class DataManager:
    def __init__(self, context_file = "/home/clpud/Levy_Brain_Agent/store_files/store_context.json"):

        self.context_file = context_file
    

    def get_store_context(self): #need_to_search: Literal["yes", "no"]
        """In this we have to get all the store context in the store"""
        try:
            with open(self.context_file, "r") as file:
                context = json.load(file)
            return context
        except Exception as e:
            print(f"Error loading store context: {e}")
            return {}
        

    def update_store_context(self, new_context: Annotated[str, "The new context we nned to write in the Store file."]):
        """In this we have to update new content in the system"""
        try:
            with open(self.context_file, "w") as file:
                json.dump(new_context, file)
        except Exception as e:
            return "please try again"

In [30]:
from langchain.tools import tool

data_manager = DataManager()
all_tools = [tool(data_manager.get_store_context)]



In [32]:
all_tools[0]

StructuredTool(name='get_store_context', description='In this we have to get all the store context in the store', args_schema=<class 'langchain_core.utils.pydantic.get_store_context'>, func=<bound method DataManager.get_store_context of <__main__.DataManager object at 0x7f95d5153e90>>)

In [52]:
from langchain.agents import AgentExecutor, create_react_agent
from langchain import hub
from langchain.prompts import PromptTemplate
from langchain.output_parsers import PydanticOutputParser
from langchain_core.utils.function_calling import convert_to_openai_function, convert_pydantic_to_openai_tool, convert_pydantic_to_openai_function


# prompt = hub.pull("hwchase17/react")
class BrainAgentOutput(BaseModel):
    final_answer: str = Field(..., description="The final answer to the command.")
    thoughts: List[str] = Field(..., description="List of thoughts before reaching the final answer.")
    actions_taken: List[str] = Field(..., description="List of actions executed.")



schema = convert_to_openai_function(BrainAgentOutput)
schema_str = json.dumps(schema).replace("{", "{{").replace("}", "}}")



react_template = f"""
Answer the following questions as best you can. You have access to the following tools:

{{tools}}
Tools input format : 
Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{{tool_names}}]
Action Input: the input to the action
Observation: the result of the action
... 
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Return the answer to the user once completed
Begin!

You are a brain agent and connected with robot. Manger will give you a command to execute. You have to generate a thought plan how to execute that plan. 
Manager Commnad: {{command}}

In this you will receive a plan from the text agent which is corrosponding the Command and a plan which is send by a robot.

Text Plan: {{text_agent_plan}}
Image Plan: {{Image_agent_plan}}


Thought: {{agent_scratchpad}}.

"""
# (this Thought/Action/Action Input/Observation can repeat N times)
local_prompt = PromptTemplate(
    input_variables=["agent_scratchpad","tool_names", "tools","command", "text_agent_plan", "Image_agent_plan"],
    template=react_template
)



agent_brain  = create_react_agent(llm, all_tools, local_prompt)
brain_executer = AgentExecutor(
    agent=agent_brain,
    tools=all_tools,
    verbose=True,
    return_intermediate_steps=True,
    max_execution_time=None,
    max_iterations=15,
)

In [47]:
schema_str

'{{"name": "BrainAgentOutput", "description": "", "parameters": {{"properties": {{"final_answer": {{"description": "The final answer to the command.", "type": "string"}}, "thoughts": {{"description": "List of thoughts before reaching the final answer.", "items": {{"type": "string"}}, "type": "array"}}, "actions_taken": {{"description": "List of actions executed.", "items": {{"type": "string"}}, "type": "array"}}}}, "required": ["final_answer", "thoughts", "actions_taken"], "type": "object"}}}}'

In [ ]:
q = "what are you doing?"
ans = brain_executer.invoke({"input": [HumanMessage(content=q)]})

In [53]:



text_agent_plan = [
                "Check the inventory levels of aisle 5",
                "Gather the necessary items to restock",
                "Move to aisle 5 and begin restocking shelves",
                "Ensure all items are faced and fronted properly",
                "Clean up any debris or empty boxes"
            ]


Image_agent_plan = [
                "Check inventory levels",
                "Order restock from warehouse",
                "Restock shelves",
                "Verify restock"
            ]

command = "restock aisle 5"
ans = brain_executer.invoke({"text_agent_plan": text_agent_plan  , "Image_agent_plan":Image_agent_plan, "command":command  })



> Entering new AgentExecutor chain...
Question: How to execute the manager's command to restock aisle 5
Thought: To execute the command, I need to understand the current state of the store, specifically the inventory levels and the layout of aisle 5. I have received two plans, a text plan and an image plan, but I need more information about the current store context to determine the best course of action.
Action: get_store_context
Action Input: None{'aisles': {'aisle_1': {'location': 'north_wing', 'shelves': {'left': [{'shelf_number': 1, 'category': 'Beverages', 'subcategories': {'Soft Drinks': [{'product': 'Cola', 'stock': 50, 'reorder_threshold': 10}, {'product': 'Lemonade', 'stock': 30, 'reorder_threshold': 15}], 'Juices': [{'product': 'Orange Juice', 'stock': 40, 'reorder_threshold': 20}, {'product': 'Apple Juice', 'stock': 25, 'reorder_threshold': 10}]}}, {'shelf_number': 2, 'category': 'Snacks', 'subcategories': {'Chips': [{'product': 'Potato Chips', 'stock': 60, 'reorder_thres

In [54]:
ans["output"]

'To restock aisle 5, I will follow these steps: \n\n1. Check the inventory levels of aisle 5, which is located in the south wing.\n2. Gather the necessary items to restock the shelves, based on the reorder thresholds for each product.\n3. Move to aisle 5 and begin restocking the shelves, ensuring that all items are faced and fronted properly.\n4. Clean up any debris or empty boxes, and verify that the restocking has been completed successfully.\n\nThis plan will ensure that aisle 5 is fully restocked and that the shelves are well-organized and easy to shop.'

In [ ]:
import json
import networkx as nx

# Sample JSON data
# data = {
#     "aisles": {
#         "aisle_1": {
#             "location": "north_wing",
#             "shelves": {
#                 "left": [
#                     {
#                         "shelf_number": 1,
#                         "category": "Beverages",
#                         "subcategories": {
#                             "Soft Drinks": [
#                                 {"product": "Cola", "stock": 50, "reorder_threshold": 10},
#                                 {"product": "Lemonade", "stock": 30, "reorder_threshold": 15}
#                             ]
#                         }
#                     }
#                 ]
#             }
#         }
#     }
# }


with open("/home/clpud/Levy_Brain_Agent/store_files/store_context.json","r") as f:
    data = json.load(f)

# Create a directed graph
G = nx.DiGraph()

# Function to build the knowledge graph
def build_knowledge_graph(G, data):
    for aisle, aisle_info in data["aisles"].items():
        G.add_node(aisle, type="aisle", location=aisle_info["location"])
        for side, shelves in aisle_info["shelves"].items():
            for shelf in shelves:
                shelf_node = f"{aisle}_shelf_{shelf['shelf_number']}_{side}"
                G.add_node(shelf_node, type="shelf", number=shelf["shelf_number"], side=side)
                G.add_edge(aisle, shelf_node)

                category = shelf["category"]
                category_node = f"{aisle}_{category}"
                G.add_node(category_node, type="category", name=category)
                G.add_edge(shelf_node, category_node)

                for subcategory, products in shelf["subcategories"].items():
                    subcategory_node = f"{aisle}_{subcategory}"
                    G.add_node(subcategory_node, type="subcategory", name=subcategory)
                    G.add_edge(category_node, subcategory_node)

                    for product in products:
                        product_node = product["product"]
                        G.add_node(
                            product_node,
                            type="product",
                            stock=product["stock"],
                            reorder_threshold=product["reorder_threshold"],
                            aisle=aisle,
                            shelf_number=shelf["shelf_number"],
                            shelf_side=side,
                            category=category,
                            subcategory=subcategory
                        )
                        G.add_edge(subcategory_node, product_node)


    return G

# Build the graph
build_knowledge_graph(data)


In [57]:

# Function to get all products on a shelf
def get_shelf_products(aisle, shelf_side, shelf_number):
    """Fetch all products from a specific shelf."""
    shelf_node = f"{aisle}_shelf_{shelf_number}_{shelf_side}"
    
    if shelf_node in G:
        products = []
        for category in G.successors(shelf_node):
            for subcategory in G.successors(category):
                for product in G.successors(subcategory):
                    product_data = G.nodes[product]
                    products.append({
                        "Product": product,
                        "Stock": product_data["stock"],
                        "Reorder Threshold": product_data["reorder_threshold"],
                        "Category": product_data["category"],
                        "Subcategory": product_data["subcategory"]
                    })
        return {"Aisle": aisle, "Shelf Side": shelf_side, "Shelf Number": shelf_number, "Products": products}
    
    return "Shelf not found"

# New Function: Get shelf details using category or product name
def get_shelf_details(identifier):
    """Retrieve aisle, shelf number, and shelf side using a category or product name."""
    if identifier in G:
        node_data = G.nodes[identifier]

        if node_data["type"] == "product":
            return {
                "Aisle": node_data["aisle"],
                "Shelf Side": node_data["shelf_side"],
                "Shelf Number": node_data["shelf_number"],
                "Category": node_data["category"],
                "Subcategory": node_data["subcategory"],
                "Stock": node_data["stock"],
                "Reorder Threshold": node_data["reorder_threshold"]
            }
        elif node_data["type"] == "category":
            shelves = []
            for subcategory in G.successors(identifier):
                for product in G.successors(subcategory):
                    product_data = G.nodes[product]
                    shelves.append({
                        "Aisle": product_data["aisle"],
                        "Shelf Side": product_data["shelf_side"],
                        "Shelf Number": product_data["shelf_number"]
                    })
            return shelves if shelves else "No shelves found for this category."
    
    return "Identifier not found in the knowledge graph."



In [61]:
# Example Queries
print(get_shelf_products("aisle_1", "left", 1))
# print(get_shelf_details("Cola"))  # Search by product name
# print(get_shelf_details("Beverages"))  # Search by category


{'Aisle': 'aisle_1', 'Shelf Side': 'left', 'Shelf Number': 1, 'Products': [{'Product': 'Cola', 'Stock': 50, 'Reorder Threshold': 10, 'Category': 'Beverages', 'Subcategory': 'Soft Drinks'}, {'Product': 'Lemonade', 'Stock': 30, 'Reorder Threshold': 15, 'Category': 'Beverages', 'Subcategory': 'Soft Drinks'}, {'Product': 'Orange Juice', 'Stock': 40, 'Reorder Threshold': 20, 'Category': 'Beverages', 'Subcategory': 'Juices'}, {'Product': 'Apple Juice', 'Stock': 25, 'Reorder Threshold': 10, 'Category': 'Beverages', 'Subcategory': 'Juices'}]}


In [62]:
print(get_shelf_details("Apple Juice"))  # Search by product name

{'Aisle': 'aisle_1', 'Shelf Side': 'left', 'Shelf Number': 1, 'Category': 'Beverages', 'Subcategory': 'Juices', 'Stock': 25, 'Reorder Threshold': 10}
